In [156]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, CSVLoader, UnstructuredExcelLoader, UnstructuredPDFLoader
import unstructured
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories import ChatMessageHistory
from sklearn.metrics import precision_score, recall_score
import time
from langchain_core.documents import Document

In [157]:
setwd = "D:/Sathish/AIML/RAG"

In [158]:
def document_load(file_path: str):
    if not os.path.exists(file_path):
        return FileNotFoundError(f"File not found: {file_path}")
    if file_path.endswith(".pdf"):
        try:
            # Try normal PDF loader first
            loader = PyPDFLoader(file_path)
            docs = loader.load()
            # If no text was extracted (scanned PDF), fall back to OCR
            if len(docs) == 0:
                loader = UnstructuredPDFLoader(file_path, mode="elements", strategy="fast", ocr=True)
                docs = loader.load()
        except Exception:
            # Fallback if PyPDFLoader fails
            loader = UnstructuredPDFLoader(file_path, mode="elements", strategy="fast", ocr=True)
            docs = loader.load()
    elif file_path.endswith(".docx"):
        loader = Docx2txtLoader(file_path)
        docs = loader.load()
    elif file_path.endswith(".csv"):
        loader = CSVLoader(file_path)
        docs = loader.load()
    elif file_path.endswith(".xlsx"):
        loader = UnstructuredExcelLoader(file_path)
        docs = loader.load()
    else:
        raise ValueError(f"Unsupported file type: {file_path}")
    for d in docs:
        d.metadata['source'] = os.path.basename(file_path)
    return docs

In [167]:
docs = []
for file in ["AI support assistant project.pdf", "Real Estate Investment Advisor.docx", "listing.csv"]:
    docs.extend(document_load(file))
print(f"Loaded {len(docs)} chunks")

Loaded 21204 chunks


In [168]:
from collections import defaultdict
def preview_doc(docs):
    grouped = defaultdict(list)
    for d in docs:
        grouped[d.metadata['source']].append(d)
    for source, chunks in grouped.items():
        print(f"\n Document {source}: len(chunks)")
        for i, d in enumerate(chunks[:3],start=1):
            print(f"Chunk {i} : {d.page_content[:20]}")

In [169]:
preview_doc(docs)


 Document AI support assistant project.pdf: len(chunks)
Chunk 1 : AI-Powered Customer 
Chunk 2 : Model Training 
Appr
Chunk 3 : Training Setup: 
• L

 Document Real Estate Investment Advisor.docx: len(chunks)
Chunk 1 : Real Estate Investme

 Document listing.csv: len(chunks)
Chunk 1 : Listing_ID: L00001
C
Chunk 2 : Listing_ID: L00002
C
Chunk 3 : Listing_ID: L00003
C


In [170]:
#Setting a chunk
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,chunk_overlap = 100)

In [171]:
chunked_docs = splitter.split_documents(docs)
print(f"Total chunks: {len(chunked_docs)}")

Total chunks: 21211


In [172]:
#Create embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [173]:
#Store chunks converted into a vector embedding
vectorstore = FAISS.from_documents(chunked_docs, embeddings)

In [174]:
#Setting retriever to fetch the relevant chunks
retriever = vectorstore.as_retriever(search_kwargs={'k':3})

In [175]:
#Initialize chat history
chat_history = ChatMessageHistory()

In [ ]:
#Initialize LLM
llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash-lite", temperature=0,api_key="")

In [178]:
#Define prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that uses context from documents."),
    ("human", "Conversation so far:\n{history}\n\nContext:\n{context}\n\nQuestion: {question}")
])

In [179]:
#Runnable pipeline with history + retriever
def ask_question(question):
    # Add user message to history
    chat_history.add_user_message(question)

    qa_chain = (
        {
            "context": lambda x: retriever.invoke(x["question"]),
            "question": lambda x: x["question"],
            "history": lambda x: "\n".join([m.content for m in chat_history.messages])
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    # Run query
    result = qa_chain.invoke({"question": question})
    # Add AI response to history
    chat_history.add_ai_message(result)
    return result

In [180]:
print(ask_question("Summarize the projects?"))

The provided documents describe two distinct projects:

1.  **Real Estate Investment Advisor:** This project focuses on predicting property prices and classifying investment potential. It involves:
    *   Creating a "Growth Rate" feature by combining City, Property Type, and Built Year using Multiple Correspondence Analysis (MCA) and then standardizing it.
    *   Classifying properties as "Good Investment" or "Not Good Investment" based on their current price relative to the median and their projected future price.

2.  **AI Support Assistant Project:** This project appears to be an AI model development effort, likely for text classification or analysis. It details several approaches:
    *   **Approaches 1 & 2:** Utilize TF-IDF or Sentence Transformer embeddings of text, along with sentiment scores, keyword presence, and urgency, trained with Logistic Regression, Gaussian Naive Bayes, and Linear SVM. Both achieved a maximum F1 score of 51%.
    *   **Approach 3 (Deep Learning):** In

In [192]:
eval_model = SentenceTransformer("all-MiniLM-L6-v2")

def evaluate_query(question, gold_answer=None, gold_chunks=None):
    start = time.time()
    # Retrieve chunks
    retrieved = retriever.invoke(question)
    retrieved_texts = [doc.page_content for doc in retrieved]
    # Build QA chain
    qa_chain = (
        {
            "context": lambda x: retrieved,
            "question": lambda x: x["question"],
            "history": lambda x: ""
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    # Run query
    answer = qa_chain.invoke({"question": question})
    latency = time.time() - start

    # Retrieval metrics
    if gold_chunks:
        relevant_retrieved = [
        chunk for chunk in retrieved_texts 
        if any(gold in chunk for gold in gold_chunks)]
        precision = len(relevant_retrieved) / len(retrieved_texts) if retrieved_texts else 0
        recall = len(relevant_retrieved) / len(gold_chunks) if gold_chunks else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision+recall) > 0 else 0
    else:
        precision = recall = f1 = None

    # Response quality
    if gold_answer:
        sim = util.cos_sim(eval_model.encode(answer), eval_model.encode(gold_answer)).item()
    else:
        sim = None
    metrics = {
        "latency_seconds": latency,
        "retrieval_precision": precision,
        "retrieval_recall": recall,
        "retrieval_f1": f1,
        "response_similarity": sim
    }
    return metrics

In [196]:
test_queries = [
    {
        "question": "Real Estate Investment Advisor project objective?",
        "gold_answer": "Predicting property prices and classifying investment opportunities",
        "gold_chunks": [
            "Real Estate Investment Advisor: Predicting Property Profitability & Future Value",
            "Construction of Classification feature: Investment type"
        ]
    },
    {
        "question": "aim of ai-powered customer support automation project?",
        "gold_answer": "Automate customer support using classification models and generative AI",
        "gold_chunks": [
            "AI-Powered Customer Support Automation using Classification Models and Generative",
            "Deployment: LSTM model integrated with Gemini AI"
        ]
    },
    {
        "question": "Is City variable is there?",
        "gold_answer": "Yes, it has New York, Los Angeles, Houston, Phoenix,Chicago",
        "gold_chunks": ["Yes",
                        "New York", "Los Angels", "Houston", "Phoenix"]
    }
]

In [197]:
results = []
for q in test_queries:
    metrics = evaluate_query(q["question"], gold_answer=q["gold_answer"], gold_chunks=q["gold_chunks"])
    results.append(metrics)

for r in results:
    print(r)

{'latency_seconds': 1.3904144763946533, 'retrieval_precision': 0.6666666666666666, 'retrieval_recall': 1.0, 'retrieval_f1': 0.8, 'response_similarity': 0.6176294088363647}
{'latency_seconds': 1.1242554187774658, 'retrieval_precision': 0.3333333333333333, 'retrieval_recall': 0.5, 'retrieval_f1': 0.4, 'response_similarity': 0.8999656438827515}
{'latency_seconds': 0.9460873603820801, 'retrieval_precision': 0.3333333333333333, 'retrieval_recall': 0.2, 'retrieval_f1': 0.25, 'response_similarity': 0.3957407772541046}
